In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("../data/processed/sales_forecast.csv", encoding="latin1")

In [ ]:
#Computing residuals
df["residual"] = df["actual"] - df["forecast"]

In [ ]:
#Cleaning missing values
df = df.dropna(subset=["residual"])

In [ ]:
#Compute Z-score of residuals
import numpy as np

df["z_score"] = (df["residual"] - df["residual"].mean()) / df["residual"].std()

In [ ]:
#Flaging anomalies
df["anomaly"] = np.abs(df["z_score"]) > 2

In [ ]:
#Classifying anomaly type
df["anomaly_type"] = "normal"

df.loc[df["anomaly"] & (df["residual"] > 0), "anomaly_type"] = "spike"
df.loc[df["anomaly"] & (df["residual"] < 0), "anomaly_type"] = "drop"

In [ ]:
#Sorting anomalies for inspection
anomalies = df[df["anomaly"]].sort_values("z_score", ascending=False)

In [ ]:
#Saving processed file
df.to_csv("data/processed/anomalies.csv", index=False)

In [ ]:
anomalies.to_csv("data/processed/anomaly_events.csv", index=False)

In [ ]:
#Adding labls
def label_reason(row):
    if row["anomaly_type"] == "spike":
        return "Unexpected surge in sales"
    elif row["anomaly_type"] == "drop":
        return "Unexpected drop in sales"
    return "Normal behavior"

df["anomaly_reason"] = df.apply(label_reason, axis=1)

In [ ]:
#visualization
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))
plt.plot(df["date"], df["actual"], label="Actual")
plt.plot(df["date"], df["forecast"], label="Forecast")

anoms = df[df["anomaly"]]
plt.scatter(anoms["date"], anoms["actual"], color="red", label="Anomaly")

plt.legend()
plt.title("Sales Anomalies vs Forecast")
plt.show()